In [1]:
# ============================================================
# Notebook : 10_precursor_sensitivity.ipynb
# Purpose  : Sensitivity of the lagged precursor signals (reframed as
#            "time-lagged leading associations / causal-path candidates",
#            not confirmed causal effects) requested in review 3.2.
#   - Lags tau = 1, 3, 6 steps (10 / 30 / 60 minutes).
#   - Controls: the category's own autoregressive term cat(t-tau) and
#     time-of-day (hour sin/cos), to absorb circadian / sleep-linked
#     common causes raised by the reviewer.
#   - Multiple testing: Benjamini-Hochberg FDR across all sensor x category
#     tests within each lag; q-values and survival reported.
#   Partial correlations use the pooled OLS t-statistic (same estimator as
#   notebook 03/08), so strength is comparable across lags.
# Design   : Single code cell. Grayscale figures, 600 dpi, png+pdf. No path print.
# ============================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"font.family": "serif", "axes.facecolor": "white",
                     "figure.facecolor": "white", "savefig.facecolor": "white",
                     "savefig.dpi": 600, "figure.dpi": 120,
                     "axes.edgecolor": "0.15", "grid.color": "0.85"})
GRAYS = ["0.20", "0.50", "0.78"]


def save_fig(fig, name):
    FIG = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()) / "results" / "figures"
    FIG.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG / f"{name}.png", dpi=600, bbox_inches="tight")
    fig.savefig(FIG / f"{name}.pdf", bbox_inches="tight")


def partial_corr_p(y, X):
    """Return partial correlation and two-sided p-value for each column of X,
    controlling for all other columns (and intercept)."""
    y = np.asarray(y, float)
    X = np.asarray(X, float)
    n, k = X.shape
    Xd = np.column_stack([np.ones(n), X])
    beta, _, _, _ = np.linalg.lstsq(Xd, y, rcond=None)
    resid = y - Xd @ beta
    dof = n - Xd.shape[1]
    s2 = (resid @ resid) / dof
    inv = np.linalg.pinv(Xd.T @ Xd)
    se = np.sqrt(np.maximum(np.diag(inv) * s2, 1e-18))
    tv = (beta / se)[1:]
    r = tv / np.sqrt(tv ** 2 + dof)
    p = 2 * stats.t.sf(np.abs(tv), dof)
    return r, p, dof


def bh_fdr(pvals):
    """Benjamini-Hochberg q-values."""
    p = np.asarray(pvals, float)
    m = len(p)
    order = np.argsort(p)
    ranked = p[order] * m / (np.arange(m) + 1)
    q = np.minimum.accumulate(ranked[::-1])[::-1]
    out = np.empty(m)
    out[order] = np.clip(q, 0, 1)
    return out


ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA, TAB = ROOT / "data", ROOT / "results" / "tables"
df = pd.read_pickle(DATA / "analysis_home.pkl")
meta = pd.read_pickle(DATA / "var_meta.pkl")
SENSORS = meta["sensor_cont"]
CATS = meta["abnormal_categories"]
CAT_KO = {"HR": "심박이상", "BR": "호흡이상", "ST": "체온이상",
          "SP": "산소포화도이상", "ER": "응급직접신호"}

df = df.copy()
df["Timestamp"] = pd.to_datetime(df["Timestamp"])
df = df.sort_values(["session_id", "Timestamp"]).reset_index(drop=True)
hour = df["Timestamp"].dt.hour + df["Timestamp"].dt.minute / 60.0
df["hour_sin"] = np.sin(2 * np.pi * hour / 24.0)
df["hour_cos"] = np.cos(2 * np.pi * hour / 24.0)

LAGS = [1, 3, 6]        # steps -> 10, 30, 60 minutes
LAG_MIN = {1: 10, 3: 30, 6: 60}

records = []
for tau in LAGS:
    lagS = df.groupby("session_id")[SENSORS].shift(tau)
    lagS.columns = [f"{s}_lag" for s in SENSORS]
    for c in CATS:
        y = df[f"cat_{c}"].astype(float)
        ar = df.groupby("session_id")[f"cat_{c}"].shift(tau).rename("ar")
        D = pd.concat([y.rename("y"), lagS, ar,
                       df[["hour_sin", "hour_cos"]]], axis=1).dropna()
        if D["y"].sum() < 20 or len(D) < 200:
            continue
        Xcols = [f"{s}_lag" for s in SENSORS] + ["ar", "hour_sin", "hour_cos"]
        r, p, dof = partial_corr_p(D["y"].values, D[Xcols].values)
        for i, s in enumerate(SENSORS):          # keep only the sensor columns
            records.append({"category": c, "category_ko": CAT_KO[c],
                            "lag_min": LAG_MIN[tau], "sensor": s,
                            "partial_corr": r[i], "p_value": p[i], "n": len(D)})

full = pd.DataFrame(records)
# BH-FDR within each lag across all sensor x category tests
full["q_value"] = np.nan
for tau_min in full["lag_min"].unique():
    m = full["lag_min"] == tau_min
    full.loc[m, "q_value"] = bh_fdr(full.loc[m, "p_value"].values)
full["survives_fdr"] = full["q_value"] < 0.05

# Top precursor per (category, lag)
idx = full.groupby(["category", "lag_min"])["partial_corr"].apply(lambda s: s.abs().idxmax())
top = full.loc[idx.values].copy()
top = top.sort_values(["category", "lag_min"]).reset_index(drop=True)
for col in ["partial_corr", "p_value", "q_value"]:
    top[col] = top[col].astype(float).round(4)
    full[col] = full[col].astype(float).round(5)

top[["category", "category_ko", "lag_min", "sensor", "partial_corr",
     "p_value", "q_value", "survives_fdr", "n"]].to_csv(
    TAB / "table_precursor_lag_sensitivity.csv", index=False, encoding="utf-8-sig")
full.to_csv(TAB / "table_precursor_lag_all.csv", index=False, encoding="utf-8-sig")

# -------- Figure: top-precursor partial corr magnitude across lags ----------
fig, ax = plt.subplots(figsize=(8.0, 4.2))
x = np.arange(len(CATS))
w = 0.26
for j, tau_min in enumerate([10, 30, 60]):
    vals, hatch = [], []
    for c in CATS:
        row = top[(top.category == c) & (top.lag_min == tau_min)]
        vals.append(abs(row["partial_corr"].values[0]) if len(row) else 0.0)
    ax.bar(x + (j - 1) * w, vals, w, label=f"{tau_min} min lag",
           color=GRAYS[j], edgecolor="0.15", linewidth=0.7)
ax.set_xticks(x)
ax.set_xticklabels(CATS)
ax.set_ylabel("|partial correlation| of top precursor")
ax.set_xlabel("abnormal category")
ax.legend(frameon=False, fontsize=8, title="time lag")
fig.tight_layout()
save_fig(fig, "fig_precursor_lag_sensitivity")
plt.close(fig)

print("=== Top precursor per category and lag (controls: AR + time-of-day; BH-FDR) ===")
print(top[["category", "lag_min", "sensor", "partial_corr", "q_value", "survives_fdr"]].to_string(index=False))
print("\nSaved: table_precursor_lag_sensitivity.csv, table_precursor_lag_all.csv, fig_precursor_lag_sensitivity")


=== Top precursor per category and lag (controls: AR + time-of-day; BH-FDR) ===
category  lag_min          sensor  partial_corr  q_value  survives_fdr
      BR       10     Temperature       -0.0496      0.0          True
      BR       30     Temperature       -0.0638      0.0          True
      BR       60     Temperature       -0.0695      0.0          True
      ER       10     Activity_IR        0.0358      0.0          True
      ER       30        Humidity       -0.0270      0.0          True
      ER       60        Humidity       -0.0274      0.0          True
      HR       10       HeartRate       -0.0535      0.0          True
      HR       30 SkinTemperature       -0.0589      0.0          True
      HR       60 SkinTemperature       -0.0599      0.0          True
      SP       10            SPO2       -0.0562      0.0          True
      SP       30            SPO2       -0.0549      0.0          True
      SP       60            SPO2       -0.0358      0.0          Tr